# Mikado Judge — Debug Visualisation

Visualises YOLO-OBB detections and the full judge pipeline on sample image pairs.
Useful for debugging thresholds and inspecting individual cases.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/mikado-judge'
WEIGHTS_PATH = f'{DRIVE_ROOT}/weights/mikado_v1_best.pt'  # Update
BEFORE_IMAGE = f'{DRIVE_ROOT}/test_images/before.jpg'     # Update
AFTER_IMAGE  = f'{DRIVE_ROOT}/test_images/after.jpg'      # Update

In [ ]:
!pip install ultralytics --quiet
!git clone https://github.com/YOUR_USERNAME/mikado-judge.git /content/mikado-judge --quiet
!pip install -e /content/mikado-judge --quiet

In [ ]:
import cv2
import yaml
from IPython.display import display
from PIL import Image
import numpy as np

from mikado.align import FrameAligner
from mikado.detect import Detector
from mikado.judge import Judge
from mikado.track import StickMatcher
from mikado.visualize import draw_judgment, draw_sticks

with open('/content/mikado-judge/configs/judge.yaml') as f:
    config = yaml.safe_load(f)

detector = Detector.from_config(WEIGHTS_PATH, config)
aligner = FrameAligner.from_config(config)
matcher = StickMatcher.from_config(config)
judge = Judge.from_config(config)

before = cv2.imread(BEFORE_IMAGE)
after  = cv2.imread(AFTER_IMAGE)

# Align
alignment = aligner.align(before, after)
print(f'Alignment inliers: {alignment.n_inliers}, success: {alignment.success}')

# Detect
before_sticks = detector.detect(before)
after_sticks  = detector.detect(alignment.aligned_frame)
print(f'Before sticks: {len(before_sticks)}')
print(f'After sticks:  {len(after_sticks)}')

# Match and judge
match_result = matcher.match(before_sticks, after_sticks)
judgment = judge.judge(match_result)

print(f'\nVerdict: {"FAULT" if judgment.fault else "OK"}')
print(f'Matched pairs: {len(judgment.all_movements)}')
print(f'Removed sticks: {len(judgment.removed_sticks)}')
print(f'Moved sticks: {len(judgment.moved_sticks)}')

In [ ]:
vis = draw_judgment(before, alignment.aligned_frame, judgment)
display(Image.fromarray(vis[:, :, ::-1]))

In [ ]:
# Print per-stick movement metrics
print('Per-stick movements:')
for i, mv in enumerate(judgment.all_movements):
    marker = '>>> TARGET' if mv.before is judgment.target_stick else ''
    flag = 'MOVED' if mv.moved else 'ok'
    print(f'  Stick {i+1} ({mv.before.class_name}): '
          f'd={mv.centroid_displacement_px:.1f}px '
          f'a={mv.angle_change_deg:.1f}° '
          f'iou={mv.iou:.3f} [{flag}] {marker}')